# 01 — Kiểm tra dữ liệu và trích xuất đặc trưng

CSV hồ sơ hiện có 37.438 dòng, 20 cột. Các cell thực hành bên dưới được giữ nguyên. Phần **Pipeline đặc trưng dùng chung** gọi code trong `src/` để tạo bảng đặc trưng và báo cáo chất lượng; chưa chia tập hoặc huấn luyện mô hình.


In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
if not (project_root / "src").is_dir():
    raise RuntimeError("Mở notebook từ thư mục gốc dự án hoặc notebooks/.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import FEATURE_COLUMNS, RANDOM_STATE
print("Project root:", project_root)


Project root: d:\Project\Social-Media-Bot-Fake-Account-Detection


In [2]:
from src.data import audit_labels

audit = audit_labels()
audit


{'file': 'twitter_human_bots_dataset.csv',
 'rows': 37438,
 'columns': ['',
  'created_at',
  'default_profile',
  'default_profile_image',
  'description',
  'favourites_count',
  'followers_count',
  'friends_count',
  'geo_enabled',
  'id',
  'lang',
  'location',
  'profile_background_image_url',
  'profile_image_url',
  'screen_name',
  'statuses_count',
  'verified',
  'average_tweets_per_day',
  'account_age_days',
  'account_type'],
 'label_counts': {'bot': 12425, 'human': 25013},
 'missing_ids': 0,
 'missing_labels': 0,
 'duplicate_ids': 0,
 'unknown_labels': [],
 'missing_profile_columns': []}

In [3]:
import sys
import pandas as pd

print("Python của notebook:", sys.executable)
print("Phiên bản pandas:", pd.__version__)

data_path = project_root / "dataset" / "twitter_human_bots_dataset.csv"
print("CSV tồn tại:", data_path.is_file())

df = pd.read_csv(data_path, dtype={"id": "string"})

print("Kích thước:", df.shape)
print("Tên cột:", df.columns.tolist())
print("Kiểu dữ liệu của id:", df["id"].dtype)
print("Phân bố nhãn:")
print(df["account_type"].value_counts(dropna=False))

df.head(3)

Python của notebook: d:\Project\Social-Media-Bot-Fake-Account-Detection\.venv\Scripts\python.exe
Phiên bản pandas: 3.1.0
CSV tồn tại: True
Kích thước: (37438, 20)
Tên cột: ['Unnamed: 0', 'created_at', 'default_profile', 'default_profile_image', 'description', 'favourites_count', 'followers_count', 'friends_count', 'geo_enabled', 'id', 'lang', 'location', 'profile_background_image_url', 'profile_image_url', 'screen_name', 'statuses_count', 'verified', 'average_tweets_per_day', 'account_age_days', 'account_type']
Kiểu dữ liệu của id: string
Phân bố nhãn:
account_type
human    25013
bot      12425
Name: count, dtype: int64


,Unnamed: 0,created_at,default_profile,default_profile_image,description,favourites_count,followers_count,friends_count,geo_enabled,id,lang,location,profile_background_image_url,profile_image_url,screen_name,statuses_count,verified,average_tweets_per_day,account_age_days,account_type
0,0,2016-10-15 21:32:11,False,False,"Blame @xaiax, Inspired by @MakingInvisible, us...",4,1589,4,False,787405734442958848,en,unknown,http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/7874121826...,best_in_dumbest,11041,False,7.870,1403,bot
1,1,2016-11-09 05:01:30,False,False,Photographing the American West since 1980. I ...,536,860,880,False,796216118331310080,en,Estados Unidos,http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/8023296328...,CJRubinPhoto,252,False,0.183,1379,human
2,2,2017-06-17 05:34:27,False,False,Scruffy looking nerf herder and @twitch broadc...,3307,172,594,True,875949740503859204,en,"Los Angeles, CA",http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/1278890453...,SVGEGENT,1001,False,0.864,1159,human


In [4]:
required_columns = [
    "id", "account_type",
    "followers_count", "friends_count", "statuses_count",
    "created_at", "description",
    "account_age_days", "average_tweets_per_day",
]

print("Số giá trị thiếu theo cột:")
print(df[required_columns].isna().sum())

bio = df["description"].astype("string")

missing_bio = bio.isna()
blank_bio = bio.notna() & bio.fillna("").str.strip().eq("")

print("Bio null:", missing_bio.sum())
print("Bio không null nhưng rỗng/chỉ có khoảng trắng:", blank_bio.sum())
print("Tổng bio null hoặc rỗng:", (missing_bio | blank_bio).sum())

Số giá trị thiếu theo cột:
id                           0
account_type                 0
followers_count              0
friends_count                0
statuses_count               0
created_at                   0
description               7257
account_age_days             0
average_tweets_per_day       0
dtype: int64
Bio null: 7257
Bio không null nhưng rỗng/chỉ có khoảng trắng: 0
Tổng bio null hoặc rỗng: 7257


In [5]:
bio_literal = pd.read_csv(
    data_path,
    usecols=["description"],
    dtype={"description": "string"},
    keep_default_na=False,
)["description"]

print("Bio là chuỗi rỗng:", bio_literal.eq("").sum())
print(
    "Bio rỗng hoặc chỉ có khoảng trắng:",
    bio_literal.str.strip().eq("").sum(),
)

print("Nội dung gốc của các bio trước đó bị nhận diện là thiếu:")
print(bio_literal.loc[df["description"].isna()].map(repr).value_counts())

Bio là chuỗi rỗng: 7256
Bio rỗng hoặc chỉ có khoảng trắng: 7256
Nội dung gốc của các bio trước đó bị nhận diện là thiếu:
description
''        7256
'None'       1
Name: count, dtype: Int64


In [6]:
df["description"] = bio_literal.mask(
    bio_literal.eq(""),
    pd.NA,
)

print("Bio thiếu sau khi xử lý:", df["description"].isna().sum())
print("Bio chứa đúng chuỗi 'None':", df["description"].eq("None").sum())
print("Số dòng:", len(df))

Bio thiếu sau khi xử lý: 7256
Bio chứa đúng chuỗi 'None': 1
Số dòng: 37438


In [7]:
duplicate_mask = df["id"].duplicated(keep=False)

print("Số dòng thuộc các ID trùng:", duplicate_mask.sum())
print("Số ID khác nhau bị trùng:", df.loc[duplicate_mask, "id"].nunique())

labels_per_id = df.groupby("id")["account_type"].nunique(dropna=False)
conflicting_ids = labels_per_id[labels_per_id > 1].index

print("Số ID có nhãn mâu thuẫn:", len(conflicting_ids))

if duplicate_mask.any():
    display(
        df.loc[duplicate_mask, ["id", "account_type"]]
        .sort_values("id")
        .head(20)
    )

Số dòng thuộc các ID trùng: 0
Số ID khác nhau bị trùng: 0
Số ID có nhãn mâu thuẫn: 0


## Pipeline đặc trưng dùng chung

Đây là cách chạy chính thức. `load_profiles()` giữ ID và bio dưới dạng chuỗi: 7.256 bio rỗng được coi là thiếu; một bio chứa văn bản `None` được giữ nguyên. Tuổi tài khoản lấy từ nguồn, không tạo `observed_at`. Các lỗi ID/nhãn làm dừng pipeline; số không hợp lệ trở thành giá trị thiếu và được thống kê, không loại dòng.


In [8]:
from src.data import load_profiles
from src.features import run_pipeline
from src.config import RAW_LABELS_PATH, FEATURES_PATH, FEATURE_SUMMARY_PATH

profiles, data_quality = load_profiles(RAW_LABELS_PATH)
print("Kích thước hồ sơ chuẩn hóa:", profiles.shape)
display(profiles[["user_id", "following_count", "account_age_days", "description", "label"]].head())
data_quality


Kích thước hồ sơ chuẩn hóa: (37438, 20)


,user_id,following_count,account_age_days,description,label
0,787405734442958848,4.0,1403.0,"Blame @xaiax, Inspired by @MakingInvisible, us...",1
1,796216118331310080,880.0,1379.0,Photographing the American West since 1980. I ...,0
2,875949740503859204,594.0,1159.0,Scruffy looking nerf herder and @twitch broadc...,0
3,756119643622735875,633.0,1489.0,Wife.Godmother.Friend.Feline Fanatic! Assistan...,0
4,464781334,116.0,3138.0,Loan coach at @mancity & Aspiring DJ,0


{'rows_read': 37438,
 'rows_retained': 37438,
 'rows_removed': 0,
 'removal_reasons': [],
 'source_columns': ['Unnamed: 0',
  'created_at',
  'default_profile',
  'default_profile_image',
  'description',
  'favourites_count',
  'followers_count',
  'friends_count',
  'geo_enabled',
  'id',
  'lang',
  'location',
  'profile_background_image_url',
  'profile_image_url',
  'screen_name',
  'statuses_count',
  'verified',
  'average_tweets_per_day',
  'account_age_days',
  'account_type'],
 'dropped_source_index_columns': ['Unnamed: 0'],
 'identity': {'missing_ids': 0,
  'duplicate_id_rows': 0,
  'duplicate_ids': 0,
  'missing_labels': 0,
  'invalid_labels': 0,
  'conflicting_ids': 0},
 'label_counts': {'human': 25013, 'bot': 12425},
 'numeric_columns': {'followers_count': {'missing': 0,
   'non_numeric': 0,
   'non_finite': 0,
   'negative': 0,
   'fractional': 0,
   'invalid_total': 0,
   'output_missing': 0,
   'valid_zero': 1255},
  'following_count': {'missing': 0,
   'non_numeric':

In [9]:
features, feature_summary = run_pipeline()
print("Đã lưu:", FEATURES_PATH)
print("Báo cáo:", FEATURE_SUMMARY_PATH)
print("Kích thước đặc trưng:", features.shape)
display(features.head())
feature_summary


Đã lưu: D:\Project\Social-Media-Bot-Fake-Account-Detection\dataset\processed\features.csv
Báo cáo: D:\Project\Social-Media-Bot-Fake-Account-Detection\reports\feature_summary.json
Kích thước đặc trưng: (37438, 6)


,user_id,follower_following_ratio,posting_frequency,account_age_days,bio_length,label
0,787405734442958848,397.25,7.869565,1403.0,129,1
1,796216118331310080,0.977273,0.182741,1379.0,160,0
2,875949740503859204,0.289562,0.863676,1159.0,81,0
3,756119643622735875,0.816746,0.889187,1489.0,97,0
4,464781334,6497.224138,1.339069,3138.0,36,0


{'data_quality': {'rows_read': 37438,
  'rows_retained': 37438,
  'rows_removed': 0,
  'removal_reasons': [],
  'source_columns': ['Unnamed: 0',
   'created_at',
   'default_profile',
   'default_profile_image',
   'description',
   'favourites_count',
   'followers_count',
   'friends_count',
   'geo_enabled',
   'id',
   'lang',
   'location',
   'profile_background_image_url',
   'profile_image_url',
   'screen_name',
   'statuses_count',
   'verified',
   'average_tweets_per_day',
   'account_age_days',
   'account_type'],
  'dropped_source_index_columns': ['Unnamed: 0'],
  'identity': {'missing_ids': 0,
   'duplicate_id_rows': 0,
   'duplicate_ids': 0,
   'missing_labels': 0,
   'invalid_labels': 0,
   'conflicting_ids': 0},
  'label_counts': {'human': 25013, 'bot': 12425},
  'numeric_columns': {'followers_count': {'missing': 0,
    'non_numeric': 0,
    'non_finite': 0,
    'negative': 0,
    'fractional': 0,
    'invalid_total': 0,
    'output_missing': 0,
    'valid_zero': 1255

In [10]:
import pandas as pd

print("Số dòng loại:", feature_summary["data_quality"]["rows_removed"])
print("CSV nguồn không đổi:", feature_summary["source"]["unchanged"])
print("ID/nhãn khớp:", feature_summary["verification"]["id_label_alignment"])
display(pd.DataFrame(feature_summary["data_quality"]["numeric_columns"]).T)
display(pd.Series(feature_summary["output"]["missing_by_feature"], name="missing"))
feature_summary["posting_frequency_comparison"]


Số dòng loại: 0
CSV nguồn không đổi: True
ID/nhãn khớp: True


,missing,non_numeric,non_finite,negative,fractional,invalid_total,output_missing,valid_zero
followers_count,0,0,0,0,0,0,0,1255
following_count,0,0,0,0,0,0,0,6410
statuses_count,0,0,0,0,0,0,0,56
account_age_days,0,0,0,0,0,0,0,0
average_tweets_per_day,0,0,0,0,0,0,0,72


follower_following_ratio    0
posting_frequency           0
account_age_days            0
bio_length                  0
Name: missing, dtype: int64

{'source_column': 'average_tweets_per_day',
 'assumed_rounding_decimals': 3,
 'absolute_tolerance': 0.0005,
 'floating_point_margin': 1e-12,
 'comparable_rows': 37438,
 'uncomparable_rows': 0,
 'within_rounding_tolerance': 37438,
 'beyond_rounding_tolerance': 0,
 'max_absolute_difference': 0.0005000000000165983,
 'mean_absolute_difference': 0.0002505923629158624,
 'interpretation': 'Comparison only: differences beyond assumed rounding tolerance are reported, not automatically treated as data errors.'}

### Cách diễn giải

- Chỉ bốn cột `FEATURE_COLUMNS` được dùng làm đầu vào mô hình; ID và label dùng để định danh/nhãn.
- Following hoặc tuổi bằng 0 dùng mẫu số 1; thiếu/âm không bị biến thành 0.
- Bio rỗng/null có độ dài 0 là giả định xử lý, vì không phân biệt được rỗng thực sự với chưa thu thập. Unicode được đếm theo code point; giữ khoảng trắng và xuống dòng.
- `posting_frequency` là trung bình bài/ngày trong vòng đời tài khoản. So sánh với nguồn bằng giả định làm tròn 3 chữ số thập phân; sai lệch được thống kê, không tự coi là lỗi.
- Tuổi là giá trị lịch sử của nguồn, không biết chính xác thời điểm quan sát. Không cập nhật theo ngày chạy.
- Công thức, giả định, số liệu thiếu/không hợp lệ và SHA-256 nguồn nằm trong `reports/feature_summary.json`.


## Công việc ở giai đoạn sau

- Tạo bộ chia tập dùng chung, giữ một tài khoản trong đúng một tập.
- EDA phục vụ quyết định đặc trưng trên train; giữ test độc lập.
- Chỉ fit imputer/scaler trên train khi triển khai mô hình.

Các công việc trên chưa được triển khai trong pipeline đặc trưng này.
